# 07 · Skills – J.A.R.V.I.S. learns a procedure 🧪

> 🧪 **Preview.** Skills and toolboxes are preview features: names and behaviour may change. This notebook uses `AIProjectClient(..., allow_preview=True)`.

A **tool** is something an agent can *call*. A **skill** is something an agent *knows how to do*: a folder with a `SKILL.md` (YAML front matter + Markdown instructions, the open [Agent Skills](https://agentskills.io) format). Skills keep rules like "how we escalate incidents" **out of every agent's prompt** and in one versioned place.

| Delivery mode | How the agent gets the skill | Status in this repo |
|---|---|---|
| **Direct injection** | skill is downloaded into the agent project; the agent code adds it to its instructions | ✅ `jarvis-hosted` (chapter 09) |
| **Agent Framework `SkillsProvider`** | the model sees each skill's name and description and loads the body with `load_skill` when it fits | ✅ `jarvis-skills` (chapter 09) |
| **Toolbox** | skill is attached to a toolbox; MCP clients read it as an MCP *resource* | ✅ shown below |
| **Prompt agent + Microsoft Copilot harness** | skill reference in the agent definition; the harness can also run the skill's scripts | 🧪 [chapter 15](15_copilot_harness.ipynb) |

Skill versions are immutable; the skill tracks a `default_version`. Deep dive: [docs/mcp-skills.md](../docs/mcp-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_replies import show_reply, escalation_checks

import re
from pathlib import Path
from azure.ai.projects.models import SkillInlineContent

skills = project(preview=True).beta.skills
SKILL_FILE = ROOT / "skills" / "stark-escalation" / "SKILL.md"
skill_md_text = SKILL_FILE.read_text(encoding="utf-8")
front_matter_text, v1_instructions = re.match(r"---\n(.*?)\n---\n(.*)", skill_md_text, re.S).groups()
skill_meta = dict(line.split(": ", 1) for line in front_matter_text.splitlines())
print(skill_meta["name"], "-", skill_meta["description"])

## 1 · Author and upload (version 1)
`name` must be lowercase letters, digits and hyphens. Creating a version creates the skill if it does not exist yet. Every run adds a version, so on a second run the number printed is higher than 1.

In [ ]:
skill_v1 = skills.create(name=skill_meta["name"], inline_content=SkillInlineContent(description=skill_meta["description"], instructions=v1_instructions.strip()))
print("created", skill_v1.name, "version", skill_v1.version)

## 2 · A new version, then promote it
Edit the policy (here: critical incidents must also name a floor), create **version 2**, test, then make it the default – agents and toolboxes that follow the default pick it up **without any code change**.

In [ ]:
v2_instructions = v1_instructions.strip() + "\n\n## Addendum\n6. For critical incidents, also state which floor of the tower is affected.\n"
skill_v2 = skills.create(name=skill_meta["name"], inline_content=SkillInlineContent(description=skill_meta["description"], instructions=v2_instructions))
print("created version", skill_v2.version)

print("before:", skills.get(name=skill_meta["name"]).default_version)
skills.update(skill_meta["name"], default_version=skill_v2.version)
print("after :", skills.get(name=skill_meta["name"]).default_version)

for v in skills.list_versions(skill_meta["name"]):
    print("  version", v.version)

## 3 · Download (direct injection)
The hosted agent (`src/jarvis-hosted`) loads every `skills/<name>/SKILL.md` into its instructions at startup. `scripts/sync_skills.py` copies the repo's `skills/` there on deploy; you can also pull a skill **from Foundry**. The next cell saves the default version (version 2 after section 2) there. `azd deploy` runs `sync_skills.py` again and replaces the folder with the repo's `skills/`, so the downloaded copy lasts only until the next deploy.

In [ ]:
import io, zipfile
data = b"".join(skills.download(skill_meta["name"]))   # download_version(name, version) pins one version
with zipfile.ZipFile(io.BytesIO(data)) as z:
    print(z.namelist())
    target = ROOT / "src" / "jarvis-hosted" / "skills" / skill_meta["name"]
    target.mkdir(parents=True, exist_ok=True)
    for n in z.namelist():
        if n.endswith("SKILL.md"):
            (target / "SKILL.md").write_bytes(z.read(n))
print("saved to", target)

## 4 · Attach to a toolbox
A **toolbox** is a named, versioned bundle of tools (and skills) behind **one MCP endpoint**. Attach the skill; omit `version` to follow the default.

In [ ]:
from azure.ai.projects.models import ToolboxSkillReference

TOOLBOX_NAME = "stark-toolbox"
toolbox = project(preview=True).toolboxes.create_version(
    name=TOOLBOX_NAME,
    description="Escalation know-how for the workshop",
    tools=[],
    skills=[ToolboxSkillReference(name=skill_meta["name"])],
)
print("toolbox version:", toolbox.version)
TOOLBOX_URL = f"{cfg('STARK_PROJECT_ENDPOINT')}/toolboxes/{TOOLBOX_NAME}/versions/{toolbox.version}/mcp?api-version=v1"
print(TOOLBOX_URL)

## 5 · Read the skill like any MCP client
Skills appear as **MCP resources** (`resources/list`, `resources/read`). Any MCP client can read them – no Foundry SDK needed. (Toolbox versions must be published/promoted to be served on the consumer endpoint; the version-specific URL above is for testing.)

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

token = credential().get_token("https://ai.azure.com/.default").token
toolbox_headers = {"Authorization": f"Bearer {token}", "Foundry-Features": "Skills=V1Preview"}

async with streamablehttp_client(TOOLBOX_URL, headers=toolbox_headers) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        resources = await session.list_resources()
        for res in resources.resources:
            print("resource:", res.uri, "-", res.name)

## 6 · Is the skill really used? A marker test
Section 5 proves the skill can be **found**. It does not prove a model **follows** it. A toolbox only offers the skill: the client has to call `resources/read` and put the text in front of the model. Agent Framework (`SkillsProvider` with `MCPSkillsSource` 🧪) does this lazily: it advertises name and description, and loads the body only when the model decides the skill fits. So test the behaviour, not the attachment.

The skill leaves two markers in an answer:

| Marker | Comes from |
|---|---|
| the final message starts with `Severity: <level>` | version 1, step 2 of the policy |
| a critical incident names a floor | the version 2 addendum, now the default |

Three steps, and only the instructions change between the last two: read the `SKILL.md` that the toolbox serves, ask a question without it, then ask the same question with it.

### Step 1 · Read the skill from the toolbox

**Goal:** get exactly the text an MCP client gets: `resources/read` on the skill's `SKILL.md`, then the front matter removed. The result, `served_instructions`, is what step 3 adds to the instructions.

**Expected:** the URI of the skill's `SKILL.md` and `has the version 2 addendum: True`. The toolbox follows the default version, and section 2 made version 2 the default.

In [ ]:
async with streamablehttp_client(TOOLBOX_URL, headers=toolbox_headers) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        skill_uris = [r.uri for r in (await session.list_resources()).resources if skill_meta["name"] in str(r.uri)]
        skill_uri = next((u for u in skill_uris if str(u).endswith("SKILL.md")), skill_uris[0])
        served_skill_md = (await session.read_resource(skill_uri)).contents[0].text
served_instructions = re.sub(r"^---\s*\n.*?\n---\s*\n", "", served_skill_md, count=1, flags=re.S).strip()   # drop the front matter
print("read", skill_uri, "-", len(served_instructions), "characters")
print("has the version 2 addendum:", "Addendum" in served_instructions)

### Step 2 · The question without the skill

**Goal:** the baseline. The model gets plain instructions and nothing from the skill.

| Input | Value |
|---|---|
| instructions | `You are J.A.R.V.I.S.` |
| question (`INCIDENT_QUESTION`) | A suit just started smoking in the hangar. What do I do? |

**Expected:** ordinary safety advice, most likely without a `Severity:` line at the start. Both outcomes are possible, so the check shows `yes` / `no` without ✅ / ❌. The floor check only looks for the word *floor* in the final message, so an answer can hit it by chance ("stay low to the floor").

`show_reply()` ([stark_replies.py](stark_replies.py)) shows the prompt, the instructions (folded), the tool calls, the whole answer and a check table. The checks look at the final message, because agents often send a preamble first. A check table looks like this:

| | |
|---|---|
| severity first (final message) | no |
| severity (final message) | none |
| names a floor (final message) | no |

In [ ]:
project_openai = openai_client()
INCIDENT_QUESTION = "A suit just started smoking in the hangar. What do I do?"
plain_instructions = "You are J.A.R.V.I.S."

plain_reply = project_openai.responses.create(model=model("chat"), instructions=plain_instructions,
                                              input=INCIDENT_QUESTION, store=False)
show_reply(INCIDENT_QUESTION, plain_reply, escalation_checks(plain_reply), instructions=plain_instructions, to=model("chat"))

### Step 3 · The same question with the skill

**Goal:** one change only: the `SKILL.md` text from step 1 goes after the plain instructions. Same model, same question.

| Input | Value |
|---|---|
| instructions | `You are J.A.R.V.I.S.`, a blank line, then `served_instructions` (step 1) |
| question | `INCIDENT_QUESTION`, unchanged |

**Expected:** the final message starts with `Severity: critical` (step 1 of the policy: anything that can explode is critical), tells you to evacuate and notify Pepper Potts, offers no workarounds, and names the affected floor (the version 2 addendum). The check shows `yes (expected yes) ✅` for both markers and `critical` for the severity. ❓ The level is the model's call, so a run may say `medium`: read the answer before you blame the skill.

In [ ]:
skill_instructions = plain_instructions + "\n\n" + served_instructions
skill_reply = project_openai.responses.create(model=model("chat"), instructions=skill_instructions,
                                              input=INCIDENT_QUESTION, store=False)
show_reply(INCIDENT_QUESTION, skill_reply,
           escalation_checks(skill_reply, expected={"severity first": True, "severity": "critical", "names a floor": True}),
           instructions=skill_instructions, to=model("chat"))

Both markers in step 3 and not in step 2: the text the toolbox serves reaches the model and changes its answer. One call is a sample, so run it a few times, or turn the marker into an evaluation criterion (chapter 10).

Use the same test on your own agent:

| Consumer of the toolbox | Is the skill used? | How to check |
|---|---|---|
| your code reads it and adds it to the instructions (steps 1 and 3) | ✅ always | marker test |
| Agent Framework `SkillsProvider` with `MCPSkillsSource` 🧪 | only when the model loads it | marker test, plus the `resources/read` call in the traces |
| prompt agent with an `MCPTool` on the toolbox URL | ❓ not documented, not tested here | marker test before you rely on it |

## 👀 In the portal
🧪 Skills and the `stark-toolbox` toolbox are preview. The agent page shows the toolbox; where skills are listed was not checked.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Nothing is deleted by default, so you can still find the skill and the toolbox in the portal. No later chapter needs them: chapter 09 deploys the repo's `skills/` files, and chapter 15 publishes its own versions of `stark-escalation`. Uncomment the lines to remove them, the toolbox first, because it refers to the skill.

In [ ]:
# project(preview=True).toolboxes.delete(TOOLBOX_NAME)   # removes the toolbox and all its versions
# skills.delete(name=skill_meta["name"])                   # removes the skill and all its versions

## 🎯 Recap
Skills move **procedures** out of prompts: author once, version, promote, deliver to many agents. Tools do things; skills teach how. To draft a new skill, run [samples/skills_provider.py](../samples/skills_provider.py) and check which skill the model loads. A skill reference on a prompt agent only works on the Microsoft Copilot harness: that is [chapter 15](15_copilot_harness.ipynb).

> 💡 Next: [08 · Agents from YAML](08_agents_from_yaml.ipynb).